In [14]:
import pandas as pd
import spatialdata as sd
from src.utils import read_obs
from datetime import datetime
from pathlib import Path
import geopandas as gpd

In [15]:
current_datetime = datetime.now()
formatted_date = current_datetime.strftime("%Y-%m-%d")

In [152]:
dfs = {"cell_dfs":{}, "tau_dfs":{}, "plaque_dfs":{}}

In [153]:
sdata_paths

[PosixPath('/data/sdata_ptau_1/1417643954_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_1/1417643942_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_1/1417643951_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_1/1417648792_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_1/1417648783_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_1/1417643362_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_1/1417644673_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_1/1417643939_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_2/1417644670_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_2/1417643365_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_2/1417648795_CAH_processed_tau.zarr'),
 PosixPath('/data/sdata_ptau_2/1417648780_CAH_processed_tau.zarr')]

In [154]:
neuropath_obs = pd.read_csv("/root/capsule/data/Supplemental Tables/Supplemental Table 7.csv", index_col = 0)

In [155]:
obs = read_obs("/root/capsule/data/combined_adata/CaH_Xenium.2026-01-07.h5ad")

# pTau DataFrame

In [156]:
sdata_paths = list(Path("/data/sdata_ptau_1").glob("*.zarr")) + list(Path("/data/sdata_ptau_2").glob("*.zarr")) 

In [157]:
sdata_paths_selected  = sorted([i.name.split("_")[0] for i in sdata_paths])

In [158]:
high_ptau_donors = neuropath_obs[(neuropath_obs['percent AT8 positive area'] >= 0.1) & (neuropath_obs.index.isin(obs['Donor ID']))].index

In [159]:
sdata_paths_derived = sorted(obs[obs['Donor ID'].isin(high_ptau_donors) & (obs['Used in analysis'])]['barcode'].unique().tolist())

In [160]:
set(sdata_paths_derived) == set(sdata_paths_selected)

True

In [167]:

for path in sdata_paths:
    barcode = path.stem.split("_")[0]
    sdata = sd.read_zarr(path)
    gpd = sd.transform(sdata['cellular_tau_boundaries'], to_coordinate_system='global')
    # gpd['cell_id'] = sdata['mapped_table'].obs['cell_id'].values.copy()
    dfs['tau_dfs'][barcode] = gpd

/tmp/ipykernel_25748/2989816845.py:3: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(path)
2026-02-06 01:05:12 | [INFO] root_attr: multiscales
2026-02-06 01:05:12 | [INFO] root_attr: omero
2026-02-06 01:05:12 | [INFO] root_attr: spatialdata_attrs
2026-02-06 01:05:12 | [INFO] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.000039878768544, 2.0], 'type': 'scale'}], 'path': '1'}, {'coordinateTransformations': [{'scale': [1.0, 4.000079757537088, 4.0], 'type': 'scale'}], 'path': '2'}, {'coordinateTransformations': [{'scale': [1.0, 8.000159515074175, 8.0], 'type': 'scale'}], 'path': '3'}, {'coordinateTransformations': [{'scale': [1.0, 16.002871729419272, 16.0], 'type': 'scale'}], 'path': '4'}]
2026-02-06 01:05:12 | [INFO] resolution: 0
2026-02-06 01:05:12 | [

2026-02-06 01:05:12 | [INFO] resolution: 3
2026-02-06 01:05:12 | [INFO]  - shape ('c', 'y', 'x') = (1, 6269, 4264)
2026-02-06 01:05:12 | [INFO]  - chunks =  ['1', '1024 (+ 125)', '1024 (+ 168)']
2026-02-06 01:05:12 | [INFO]  - dtype = float64
2026-02-06 01:05:12 | [INFO] resolution: 4
2026-02-06 01:05:12 | [INFO]  - shape ('c', 'y', 'x') = (1, 3134, 2132)
2026-02-06 01:05:12 | [INFO]  - chunks =  ['1', '1024 (+ 62)', '1024 (+ 84)']
2026-02-06 01:05:12 | [INFO]  - dtype = float64
2026-02-06 01:05:12 | [INFO] root_attr: multiscales
2026-02-06 01:05:12 | [INFO] root_attr: omero
2026-02-06 01:05:12 | [INFO] root_attr: spatialdata_attrs
2026-02-06 01:05:12 | [INFO] root_attr: multiscales
2026-02-06 01:05:12 | [INFO] root_attr: omero
2026-02-06 01:05:12 | [INFO] root_attr: spatialdata_attrs
2026-02-06 01:05:12 | [INFO] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.000039878768544, 2.0], 't

In [168]:
tau_df = pd.concat(dfs['tau_dfs']).drop('label', axis = 1).reset_index().rename(columns = {"level_0":"barcode"})
# plaque_df = pd.concat(dfs['plaque_dfs']).drop('label', axis = 1).reset_index().rename(columns = {"level_0":"barcode"})

In [169]:
tau_df.to_parquet(f"/results/seaad_cah_tau_polygons.{formatted_date}.parquet")

# Plaque DataFrame

In [170]:
neuropath_obs = pd.read_csv("/root/capsule/data/Supplemental Tables/Supplemental Table 7.csv", index_col = 0)

In [171]:
high_plaque_donors = neuropath_obs[neuropath_obs['percent 6e10 positive area'] > 0.01].index

In [172]:
obs = read_obs("/root/capsule/data/combined_adata/CaH_Xenium.2026-01-07.h5ad")

In [173]:
section_barcodes = obs[(obs['Donor ID'].isin(high_plaque_donors)) & (obs['Used in analysis'])]['barcode'].unique()

In [174]:
section_barcodes

['1417648780', '1417648795', '1417643954', '1417649444', '1417644670', ..., '1417649450', '1417643942', '1417649441', '1417648783', '1417648792']
Length: 16
Categories (35, object): ['1417642334', '1417642340', '1417642343', '1417642733', ..., '1417649447', '1417649450', '1417649873', '1417649876']

In [175]:
for barcode in section_barcodes:
    path = Path(f"/root/capsule/data/spatialdata/{barcode}_with_vs200_CAH_processed.zarr")
    sdata = sd.read_zarr(path)
    gpd = sd.transform(sdata['plaque_boundaries'], to_coordinate_system='global')
    # gpd['cell_id'] = sdata['mapped_table'].obs['cell_id'].values.copy()
    dfs['plaque_dfs'][barcode] = gpd

/tmp/ipykernel_25748/2479214617.py:3: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(path)
2026-02-06 01:07:39 | [INFO] root_attr: multiscales
2026-02-06 01:07:39 | [INFO] root_attr: omero
2026-02-06 01:07:39 | [INFO] root_attr: spatialdata_attrs
2026-02-06 01:07:39 | [INFO] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.000049679566794, 2.0000540365286934], 'type': 'scale'}], 'path': '1'}, {'coordinateTransformations': [{'scale': [1.0, 3.999900645802285, 4.000540423692175], 'type': 'scale'}], 'path': '2'}, {'coordinateTransformations': [{'scale': [1.0, 7.999006556725611, 8.00108084738435], 'type': 'scale'}], 'path': '3'}, {'coordinateTransformations': [{'scale': [1.0, 16.00755467196819, 16.0021616947687], 'type': 'scale'}], 'path': '4'}]
2026-02-06 01

2026-02-06 01:07:39 | [INFO] resolution: 1
2026-02-06 01:07:39 | [INFO]  - shape ('c', 'y', 'x') = (3, 20129, 18506)
2026-02-06 01:07:39 | [INFO]  - chunks =  ['1', '1024 (+ 673)', '1024 (+ 74)']
2026-02-06 01:07:39 | [INFO]  - dtype = uint16
2026-02-06 01:07:39 | [INFO] resolution: 2
2026-02-06 01:07:39 | [INFO]  - shape ('c', 'y', 'x') = (3, 10065, 9252)
2026-02-06 01:07:39 | [INFO]  - chunks =  ['1', '1024 (+ 849)', '1024 (+ 36)']
2026-02-06 01:07:39 | [INFO]  - dtype = uint16
2026-02-06 01:07:39 | [INFO] resolution: 3
2026-02-06 01:07:39 | [INFO]  - shape ('c', 'y', 'x') = (3, 5033, 4626)
2026-02-06 01:07:39 | [INFO]  - chunks =  ['1', '1024 (+ 937)', '1024 (+ 530)']
2026-02-06 01:07:39 | [INFO]  - dtype = uint16
2026-02-06 01:07:39 | [INFO] resolution: 4
2026-02-06 01:07:39 | [INFO]  - shape ('c', 'y', 'x') = (3, 2515, 2313)
2026-02-06 01:07:39 | [INFO]  - chunks =  ['1', '1024 (+ 467)', '1024 (+ 265)']
2026-02-06 01:07:39 | [INFO]  - dtype = uint16
2026-02-06 01:07:39 | [INFO] ro

In [176]:
plaque_df = pd.concat(dfs['plaque_dfs']).drop('label', axis = 1).reset_index().rename(columns = {"level_0":"barcode"})

In [177]:
plaque_df.to_parquet(f"/results/seaad_cah_plaque_polygons.{formatted_date}.parquet")

# Cell DataFrame

In [178]:
section_barcodes = obs[obs['Used in analysis']]['barcode'].unique()

In [179]:
for barcode in section_barcodes:
    path = Path(f"/root/capsule/data/spatialdata/{barcode}_with_vs200_CAH_processed.zarr")
    sdata = sd.read_zarr(path)
    gpd = sd.transform(sdata['cell_boundaries'], to_coordinate_system='global')
    gpd['cell_id'] = sdata['mapped_table'].obs['cell_id'].copy()
    dfs['cell_dfs'][barcode] = gpd

/tmp/ipykernel_25748/2486093872.py:3: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(path)
2026-02-06 01:08:33 | [INFO] root_attr: multiscales
2026-02-06 01:08:33 | [INFO] root_attr: omero
2026-02-06 01:08:33 | [INFO] root_attr: spatialdata_attrs
2026-02-06 01:08:33 | [INFO] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.0, 2.0000638202820857], 'type': 'scale'}], 'path': '1'}, {'coordinateTransformations': [{'scale': [1.0, 4.000264970853206, 4.000382946132244], 'type': 'scale'}], 'path': '2'}, {'coordinateTransformations': [{'scale': [1.0, 8.000529941706413, 8.000765892264488], 'type': 'scale'}], 'path': '3'}, {'coordinateTransformations': [{'scale': [1.0, 16.001059883412825, 16.00561797752809], 'type': 'scale'}], 'path': '4'}]
2026-02-06 01:08:33 | [I

In [180]:
cell_df = pd.concat(dfs['cell_dfs']).reset_index().rename(columns = {'level_0':'barcode', 'level_1':'label'})

In [181]:
cell_df.index = cell_df['label'] + "_" + cell_df['barcode']

In [182]:
obs.index = obs['cell_id'].astype(str) + "_" + obs['barcode'].astype(str)

In [183]:
cluster_columns = ['Neighborhood', 'Subclass', 'Supertype']

In [184]:
cell_df = cell_df.merge(obs[cluster_columns], left_index = True, right_index = True, how = "left")

In [185]:
cell_df.to_parquet(f"/results/seaad_cah_cell_segmentation_polygons.{formatted_date}.parquet")